# Lab 18: Does Selling Options Earn a Reliable Premium?

*Financial Engineering and the Evidence*, Chapter 18.

**Claim on trial:** implied volatility exceeds the volatility that follows, on average, so option sellers, for example covered-call and put-writing strategies, earn a reliable premium.

You will measure the gap between the VIX and the volatility that followed, rebuild put-writing and covered-call strategies from the VIX, compare them with the market (and with Cboe's indices if you download them), and ask how much of their return is a volatility premium and how much is plain equity exposure. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab18

SOURCE = "french"     # "french" downloads daily factors and the VIX; "sim" uses fake data to test the code
FILES = {
    "ff3": None,      # e.g. "data/F-F_Research_Data_Factors_daily_CSV.zip"
    "vix": None,      # e.g. "data/VIXCLS.csv" from FRED
    "put": None,      # optional: Cboe PUT index history CSV
    "bxm": None,      # optional: Cboe BXM index history CSV
}
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab18.load(SOURCE, FILES)
SIM = d["simulated"]
print("VIX from", d["vix"].index[0].date(), "| Cboe series:", list(d["cboe"]), "| simulated:", SIM)

## Part A: The variance risk premium

**Predict first:** on average, by how many volatility points does the VIX exceed the volatility of the following month? In what share of months is it higher?

In [ ]:
a = lab18.part_a(d)
lab18.fig_vix(a, OUT, SIM)
lab18.fig_gap(a, OUT, SIM)
display(Image(OUT / "fig18_1_vix_realized.png"))
display(Image(OUT / "fig18_2_gap.png"))
a["summary"]

## Part B: Option-writing strategies

The rebuilt strategies sell a one-month option at each month end, priced by Black–Scholes at the VIX, and hold it to the next month end. An at-the-money put-write and an at-the-money covered call are the same position (put–call parity), so the covered call shown sells a call 2% out of the money.

**Predict first:** will the put-write have a higher or lower Sharpe ratio than the market? A higher or lower average return?

In [ ]:
b = lab18.part_b(d)
lab18.fig_cumulative(b, OUT, SIM)
display(Image(OUT / "fig18_3_cumulative.png"))
cols = ["strategy", "sample", "excess_annual", "vol_annual", "sharpe", "skewness", "max_drawdown", "beta", "alpha_annual", "alpha_t"]
b["table"][cols].round(3)

## Part C: Stress tests

**Predict first:** in the worst market months, does selling options cushion the fall? Will the skew-aware measure (MPPM) rank the strategies the same way as the Sharpe ratio?

In [ ]:
c = lab18.part_c(b)
lab18.fig_payoff(b, OUT, SIM)
display(Image(OUT / "fig18_4_payoff.png"))
c["crisis"]

In [ ]:
c["measures"].round(3)

## Part D: Equity exposure versus the volatility premium

**Predict first:** what share of the put-write's excess return comes from its market exposure? How does a pure short-variance position compare?

In [ ]:
dd = lab18.part_d(d, b, a)
lab18.fig_decomposition(dd, OUT, SIM)
display(Image(OUT / "fig18_5_decomposition.png"))
display(dd["decomposition"].round(3))
dd["var_swap"]

In [ ]:
# Exercise: the VIX includes a skew premium, so it overstates at-the-money implied volatility.
# Rebuild the strategies with options priced 2 points below the VIX and compare.
b2 = lab18.part_b(d, iv_shift=-2.0)
b2["table"][b2["table"]["sample"] == "Common period"][cols].round(3)

## Your verdict

1. Is implied volatility reliably above realized volatility? How often, and by how much?
2. Do put-writing and covered calls beat the market once you account for beta and skewness?
3. How much of an option-income strategy's return is equity, and how much is a volatility premium?

Write one paragraph, using the chapter's verdict scale: survives, weakens, fails or provisional.